In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import json
from IPython.display import display
import pycountry
import pycountry_convert as pc

from emu_renewal.inputs import DATA_PATH, get_country_vars

In [ ]:
included = json.load(open(DATA_PATH / "config/included.json", "r"))

In [ ]:
def get_pre_alpha_vars(c, min_samples=5, end_date=datetime(2021, 6, 30)):
    var_data = get_country_vars(c)
    var_data = var_data[var_data.index < end_date]
    var_data = var_data[var_data.sum(axis=1) >= min_samples]
    pre_alpha_cols = ["20A.EU1", "20A.EU2", "20B.S.732A", "21C.Epsilon"]
    avail_pre_alpha = [c for c in pre_alpha_cols if c in var_data.columns]
    pre_alpha_vals = var_data[avail_pre_alpha].sum(axis=1)
    totals = var_data.sum(axis=1)
    country_df = pd.DataFrame(
        {
            "pre_alpha": pre_alpha_vals,
            "totals": totals,
            "pre_alpha_prop": pre_alpha_vals / totals,
        }
    )
    return country_df[(0.0 < country_df["pre_alpha_prop"]) & (country_df["pre_alpha_prop"] < 1.0)]

def get_sufficient_pre_alpha_vars(countries, min_obs=5):
    all_data = {}
    for c in countries:
        data = get_pre_alpha_vars(c)
        if len(data) > min_obs:
            all_data[c] = data
    return all_data

def get_continent_pre_alpha_vars(data):
    continents = ["NA", "SA", "AS", "EU"]
    cont_data_dict = {}
    for cont in continents:
        cont_data = pd.DataFrame()
        for c in data:
            if pc.country_alpha2_to_continent_code(pycountry.countries.lookup(c).alpha_2) == cont:
                cont_data = cont_data.add(data[c], fill_value=0.0)
        cont_data["pre_alpha_prop"] = cont_data["pre_alpha"] / cont_data["totals"]
        cont_data_dict[cont] = cont_data
    return cont_data_dict

def get_country_var_prop(c, country_data, cont_data):
    continent = pc.country_alpha2_to_continent_code(pycountry.countries.lookup(c).alpha_2)
    if c in country_data:
        return country_data[c]
    elif continent == "AF":
        return None
    else:
        return cont_data[continent]

In [ ]:
country_data = get_sufficient_pre_alpha_vars(included)
cont_data = get_continent_pre_alpha_vars(country_data)

In [ ]:
var_data = {}
for c in included:
    var_data[c] = get_country_var_prop(c, country_data, cont_data)

In [ ]:
for c in var_data:
    if var_data[c] is not None:
        print(var_data[c]["pre_alpha_prop"])

In [ ]:
c_var_data

In [ ]:
for i in range(1, c_var_data.shape[0]):
    this_val = c_var_data["pre_alpha_prop"].iloc[i]
    prev_val = c_var_data["pre_alpha_prop"].iloc[i - 1]
    print(this_val > prev_val)

In [ ]:
for c, data in var_data.items():
    if data is not None:
        print(var_data[c].is_monotonic_decreasing)